# 1. Sample selection

Builds the market-level sample used throughout the paper (Section 3.1).

**Input:** `polymarket_closed_markets.parquet`, every closed market from Polymarket's Gamma API
(combined from four raw parts on first run).
**Output:** `finalsample.parquet`, the 50,000 binary markets with the highest CLOB trading volume.

Steps:
1. Keep binary (Yes/No) markets only.
2. Keep the 50,000 markets with the highest CLOB volume.

Further cleaning (missing volume or close time, markets that did not resolve cleanly to 0 or 1)
happens in `panel_construction.ipynb`.

In [2]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../raw")
RAW_PARTS = [RAW_DIR / f"polymarket_closed_markets_part_{i}.parquet" for i in range(1, 5)]
COMBINED_PATH = Path("polymarket_closed_markets.parquet")
OUTPUT_PATH = Path("finalsample.parquet")

BINARY_OUTCOMES = '["Yes", "No"]'  # how the Gamma API encodes a binary market's outcomes
VOLUME_COL = "volumeClob"          # volume traded on the central limit order book, in USD
N_MARKETS = 50_000

## Load the raw data

I had to split the Gamma API download into 4 parts, so we'll combine them into
one here. 

In [ ]:
def load_closed_markets() -> pd.DataFrame:
    """Return every closed market, combining the raw parts only on first run."""
    if not COMBINED_PATH.exists():
        parts = [pd.read_parquet(path) for path in RAW_PARTS]
        combined = pd.concat(parts, ignore_index=True)
        combined.to_parquet(COMBINED_PATH, index=False, compression="zstd")
    return pd.read_parquet(COMBINED_PATH)


markets = load_closed_markets()
selection_log = {"Closed markets": len(markets)}

## Step 1: Binary markets

Restricting to Yes/No markets means a single price $p$ describes the market's belief,
with $p = \mathbb{P}(Y=1)$ and $1-p = \mathbb{P}(Y=0)$.

In [4]:
binary = markets[markets["outcomes"] == BINARY_OUTCOMES]
selection_log["Binary markets"] = len(binary)

## Step 2: Top markets by volume

Low-volume markets are thinly traded, so their prices are mostly stale quotes rather than
aggregated beliefs. We keep the 50,000 markets with the most CLOB volume.

In [5]:
sample = binary.sort_values(VOLUME_COL, ascending=False).head(N_MARKETS)
selection_log[f"Top {N_MARKETS:,} by CLOB volume"] = len(sample)

assert len(sample) == N_MARKETS
assert sample["id"].is_unique

cutoff_volume = sample[VOLUME_COL].min()
cutoff_percentile = (binary[VOLUME_COL] <= cutoff_volume).mean() * 100
print(f"Volume of the smallest included market: ${cutoff_volume:,.0f}")
print(f"Percentile of that market among all binary markets: {cutoff_percentile:.1f}")

Volume of the smallest included market: $52,427
Percentile of that market among all binary markets: 71.5


## Summary

The selection log reproduces the counts reported in Section 3.1. The negative-risk split
is reported here because Section 4.3 compares the two groups.

In [6]:
pd.Series(selection_log, name="Markets").to_frame()

,Markets
Closed markets,1753814
Binary markets,603409
"Top 50,000 by CLOB volume",50000


In [7]:
sample["negRisk"].value_counts(dropna=False).rename("Markets").to_frame()

,Markets
negRisk,
True,31190
False,18234
None,576


## Save

In [8]:
sample.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(sample):,} markets to {OUTPUT_PATH}")

Saved 50,000 markets to finalsample.parquet
